# Chapter 7a — Classification Model Evaluation
**MADT6004 · Brew Lab BKK case**

For an imbalanced problem like *Comeback 50* (only ~11% respond), accuracy is a bad scorecard. The right tools:

- **Confusion matrix** — TP, FP, FN, TN
- **Precision / Recall / F1** — what each error costs you
- **ROC curve & AUC** — ranking quality across all thresholds
- **Lift chart** — for targeting decisions: how many responders do you catch in the top-decile?


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_curve, auc, accuracy_score)

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Re-build the model from Chapter 6

In [ ]:
feat = pd.read_sql("""
SELECT c.customer_id, c.acquisition_channel,
       COUNT(t.order_id) AS visits,
       COALESCE(SUM(t.total), 0) AS lifetime_spend,
       COALESCE(AVG(t.total), 0) AS avg_ticket
FROM customers c
LEFT JOIN transactions t ON t.customer_id = c.customer_id
GROUP BY c.customer_id
""", conn)
resp = pd.read_sql("SELECT customer_id, responded FROM campaign_responses", conn)
df = feat.merge(resp, on="customer_id", how="inner")

X = pd.get_dummies(df[["acquisition_channel","visits","lifetime_spend","avg_ticket"]], drop_first=True)
y = df["responded"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

lr = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
y_prob = lr.predict_proba(X_te_s)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)


## 3. Confusion matrix and headline metrics

In [ ]:
cm = confusion_matrix(y_te, y_pred)
print("Confusion matrix:\n", cm)
print(f"\nAccuracy : {accuracy_score(y_te, y_pred):.3f}")
print(f"Precision: {precision_score(y_te, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_te, y_pred):.3f}")
print(f"F1       : {f1_score(y_te, y_pred):.3f}")
print(f"\nBaseline (always predict 0): accuracy = {1 - y_te.mean():.3f}")


## 4. ROC curve and AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_te, y_prob)
auc_val = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#0891B2", lw=2, label=f"AUC = {auc_val:.3f}")
ax.plot([0, 1], [0, 1], color="#9CA3AF", lw=1, ls="--", label="random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC — Comeback 50 model"); ax.legend()
plt.tight_layout(); plt.show()


## 5. Lift chart by decile
A lift chart answers: "If we score everyone and target the top X%, how much better than random are we?"

In [ ]:
df_te = pd.DataFrame({"y": y_te.values, "p": y_prob})
df_te["decile"] = pd.qcut(df_te["p"].rank(method="first"), 10, labels=False)
df_te["decile"] = 10 - df_te["decile"]   # 1 = top decile
lift = df_te.groupby("decile").agg(rate=("y", "mean"), n=("y","size"))
overall = y_te.mean()
lift["lift"] = lift["rate"] / overall

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lift.index.astype(str), lift["lift"], color="#0891B2")
ax.axhline(1, color="#9CA3AF", lw=0.8)
ax.set_xlabel("Decile (1 = highest model score)")
ax.set_ylabel("Lift vs. base rate")
ax.set_title(f"Lift chart  —  base response rate = {overall:.1%}")
plt.tight_layout(); plt.show()
print(lift)


## Discussion prompts
1. Accuracy is high but recall is low. Why? Should Khun Ploy care?
2. If a voucher costs 50 baht and a responder is worth 300, what threshold would you pick?
3. The lift in decile 1 is the most actionable number on the page. Why?
